# Kaggle Llama AI Relevance Classification

Run the ResearchLanka AI relevance classifier on Kaggle using local Llama through Ollama. This notebook assumes you have added the Kaggle dataset that contains `ai_llm_5000_candidates.csv`.

## 1. Install Ollama

Kaggle sometimes needs `zstd` before the Ollama installer can extract its files.

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start Ollama Server

In [ ]:
import subprocess
import time

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(10)
print("Ollama server started")

## 3. Pull Llama Model

This notebook uses `llama3.1:8b`. It is slower than the 3B model, so GPU runtime is recommended.

In [ ]:
MODEL_NAME = "llama3.1:8b"

!ollama pull llama3.1:8b
!ollama list

## 4. Clone ResearchLanka AI Repo

In [ ]:
from pathlib import Path
import shutil

repo_dir = Path("/kaggle/working/researchlanka-ai")

%cd /kaggle/working

if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone --branch feature/ai-paper-filtering https://github.com/krish-anu/researchlanka-ai.git /kaggle/working/researchlanka-ai

%cd /kaggle/working/researchlanka-ai/backend
!ls scripts/ai_relevance

## 5. Install Python Requirements

The red dependency resolver warnings in Kaggle are usually not fatal for this AI relevance script.

In [ ]:
!pip install -q -r requirements.txt

## 6. Copy Uploaded 5k Candidate Dataset

Add your Kaggle dataset in the right sidebar first. This cell searches `/kaggle/input` for the candidate CSV.

In [ ]:
from pathlib import Path
import shutil

matches = list(Path("/kaggle/input").rglob("ai_llm_5000_candidates.csv"))
print("Found candidate files:")
for match in matches:
    print(" -", match)

if not matches:
    raise FileNotFoundError("Could not find ai_llm_5000_candidates.csv under /kaggle/input. Add the dataset first.")

src = matches[0]
dst = Path("data/processed/ai/ai_llm_5000_candidates.csv")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)

print("Copied:", src, "->", dst)

## 7. Quick Ollama Test

In [ ]:
!ollama run llama3.1:8b "Return one word only: READY"

## 8. Run First 100 Predictions

Use `--resume` if this cell is interrupted or if the output file already exists.

In [ ]:
import os

os.environ["AI_LLM_PROVIDER"] = "ollama"
os.environ["AI_LLM_MODEL"] = "llama3.1:8b"
os.environ["AI_PROMPT_VERSION"] = "v2"
os.environ["OLLAMA_SEED"] = "42"
os.environ["GEMINI_TIMEOUT_SECONDS"] = "300"
os.environ["GEMINI_MAX_RETRIES"] = "1"

!python scripts/ai_relevance/run_gemini_ai_relevance.py \
  --input data/processed/ai/ai_llm_5000_candidates.csv \
  --output data/processed/ai/ai_llm_first_100_predictions_ollama_kaggle.csv \
  --selected-ids-output data/processed/ai/ai_llm_first_100_ids_ollama_kaggle.csv \
  --limit 100 \
  --log-level INFO

## 9. Resume Command

Run this only if the previous cell stopped before finishing.

In [ ]:
!python scripts/ai_relevance/run_gemini_ai_relevance.py \
  --input data/processed/ai/ai_llm_5000_candidates.csv \
  --output data/processed/ai/ai_llm_first_100_predictions_ollama_kaggle.csv \
  --selected-ids-output data/processed/ai/ai_llm_first_100_ids_ollama_kaggle.csv \
  --limit 100 \
  --resume \
  --log-level INFO

## 10. Check Results

In [ ]:
import pandas as pd

predictions_path = "data/processed/ai/ai_llm_first_100_predictions_ollama_kaggle.csv"
df = pd.read_csv(predictions_path)

print(df["ai_llm_status"].value_counts(dropna=False))
print(df.loc[df["ai_llm_status"].eq("success"), "ai_llm_label"].value_counts(dropna=False))
print("rows:", len(df))

df.head(20)

## 11. Export Human Review Queue

This creates a review dataset even when Llama does not output the `REVIEW` label. It includes failed/time-out rows, low-confidence rows, borderline rows, and likely fuzzy/TOPSIS/MCDM false positives.

In [ ]:
!python scripts/ai_relevance/export_human_review_sample.py \
  --input data/processed/ai/ai_llm_first_100_predictions_ollama_kaggle.csv \
  --output data/processed/ai/ai_llm_first_100_review_queue_ollama_kaggle.csv \
  --sample-size 100 \
  --confidence-threshold 0.75

review_df = pd.read_csv("data/processed/ai/ai_llm_first_100_review_queue_ollama_kaggle.csv")
print(review_df["review_reason"].value_counts(dropna=False))
print("review rows:", len(review_df))
review_df.head(20)

## 12. Zip Outputs For Download

In [ ]:
!zip -r /kaggle/working/ollama_first_100_results.zip \
  data/processed/ai/ai_llm_first_100_predictions_ollama_kaggle.csv \
  data/processed/ai/ai_llm_first_100_ids_ollama_kaggle.csv \
  data/processed/ai/ai_llm_first_100_review_queue_ollama_kaggle.csv

print("Download: /kaggle/working/ollama_first_100_results.zip")